# MedTube Segmentation — YOLOv9c Training on Colab GPU

Trains `yolov9c-seg` on the MedTube dataset using a Colab GPU (T4 free / A100 Pro).
Weights are saved to Google Drive so they survive session disconnects.

**Before running:**
1. Runtime → Change runtime type → **GPU** (T4 or A100)
2. Fill in your Roboflow API key in Cell 3
3. Adjust `BATCH` if you get OOM errors

In [ ]:
# Cell 1 — Install dependencies
!pip install ultralytics roboflow -q

In [ ]:
# Cell 2 — Check GPU
!nvidia-smi
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0)}")

In [ ]:
# Cell 3 — Mount Google Drive (weights will be saved here)
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_SAVE_DIR = "/content/drive/MyDrive/medtube_runs"
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
print(f"Saving runs to: {DRIVE_SAVE_DIR}")

In [ ]:
# Cell 4 — Download dataset from Roboflow
# Get your API key from: https://app.roboflow.com → Settings → Roboflow API
ROBOFLOW_API_KEY = "YOUR_API_KEY_HERE"  # <-- fill in

from roboflow import Roboflow
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("tadeass-workspace").project("medtube-2")
dataset = project.version(1).download("yolov8", location="/content/medtube")

DATA_YAML = dataset.location + "/data.yaml"
print(f"Dataset downloaded to: {dataset.location}")
print(f"data.yaml: {DATA_YAML}")

In [ ]:
# Cell 5 — Training configuration
MODEL      = "yolov9c-seg.pt"   # pretrained COCO checkpoint (auto-downloaded)
EPOCHS     = 100
IMGSZ      = 640
BATCH      = 16   # T4: 16  |  A100: 32
PATIENCE   = 20
WORKERS    = 2
RUN_NAME   = "YOLOv9c-seg"

In [ ]:
# Cell 6 — Train
from ultralytics import YOLO

model = YOLO(MODEL)
results = model.train(
    data     = DATA_YAML,
    epochs   = EPOCHS,
    imgsz    = IMGSZ,
    batch    = BATCH,
    patience = PATIENCE,
    workers  = WORKERS,
    device   = 0,
    project  = DRIVE_SAVE_DIR,
    name     = RUN_NAME,
    exist_ok = True,
    # Mild augmentation — same preset as local runs
    hsv_h         = 0.01,
    hsv_s         = 0.4,
    hsv_v         = 0.2,
    degrees       = 20.0,
    translate     = 0.05,
    scale         = 0.25,
    shear         = 2.0,
    perspective   = 0.0001,
    flipud        = 0.0,
    fliplr        = 0.5,
    mosaic        = 0.2,
    erasing       = 0.2,
    auto_augment  = "randaugment",
)

BEST_WEIGHTS = f"{DRIVE_SAVE_DIR}/{RUN_NAME}/weights/best.pt"
print(f"\nTraining complete. Best weights: {BEST_WEIGHTS}")

In [ ]:
# Cell 7 — Evaluate on test split
model = YOLO(BEST_WEIGHTS)
test_results = model.val(
    data   = DATA_YAML,
    split  = "test",
    imgsz  = IMGSZ,
    batch  = BATCH,
    device = 0,
    plots  = True,
    project = DRIVE_SAVE_DIR,
    name   = f"{RUN_NAME}-test",
    exist_ok = True,
)

print("\n--- Test-split results ---")
print(f"Box  mAP50={test_results.box.map50:.4f}  mAP50-95={test_results.box.map:.4f}")
print(f"Mask mAP50={test_results.seg.map50:.4f}  mAP50-95={test_results.seg.map:.4f}")
for i, name in test_results.names.items():
    print(f"  {name}: Mask mAP50={test_results.seg.ap50[i]:.3f}")

In [ ]:
# Cell 8 — Download best weights to local machine
from google.colab import files
files.download(BEST_WEIGHTS)